# Component 6: FiLM Conditioning (Feature-wise Linear Modulation)

FiLM is the mechanism that tells the U-Net **what to do**. Without conditioning, the U-Net would just denoise blindly. With FiLM, it denoises *toward the right action trajectory* for the current observation.

## The FiLM Equation

$$h' = \gamma \cdot h + \beta$$

Where:
- **h** = hidden features (intermediate activations in the U-Net)
- **gamma** = learned scale factor (amplifies or suppresses features)
- **beta** = learned shift factor (adjusts the baseline)
- **h'** = conditioned features

Both gamma and beta are **generated from the conditioning signal** (timestep embedding + observation encoding). This means the same U-Net layer behaves differently depending on:
1. What the robot currently sees (observation)
2. How noisy the current sample is (diffusion timestep)

## Intuition

Think of FiLM like an audio equalizer:
- **Scale (gamma):** Like volume knobs for different frequency bands -- amplify important features, silence irrelevant ones
- **Shift (beta):** Like adding a DC offset -- shifts the "default" activation up or down

In [ ]:
!pip install -q torch torchvision matplotlib numpy 2>&1 | tail -3

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# ============================================================
# Build a 1D U-Net with FiLM conditioning from scratch
# (so we can inspect FiLM layers directly)
# ============================================================

class SinusoidalPosEmb(nn.Module):
    """Sinusoidal positional embedding for diffusion timesteps."""
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, x):
        half_dim = self.dim // 2
        emb = torch.log(torch.tensor(10000.0)) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=x.device, dtype=torch.float32) * -emb)
        emb = x.unsqueeze(-1).float() * emb.unsqueeze(0)
        emb = torch.cat([emb.sin(), emb.cos()], dim=-1)
        return emb


class ConditionalResidualBlock1D(nn.Module):
    """Residual block with FiLM conditioning for 1D sequences.
    
    This is where FiLM lives! The cond_encoder generates gamma and beta
    from the conditioning vector, which are used to modulate the hidden features.
    """
    def __init__(self, in_channels, out_channels, cond_dim):
        super().__init__()
        self.blocks = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=3, padding=1),
                nn.GroupNorm(8, out_channels),
                nn.Mish(),
            ),
            nn.Sequential(
                nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=1),
                nn.GroupNorm(8, out_channels),
                nn.Mish(),
            ),
        ])
        # FiLM: conditioning -> gamma, beta
        self.cond_encoder = nn.Sequential(
            nn.Mish(),
            nn.Linear(cond_dim, out_channels * 2),
        )
        self.residual_conv = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

    def forward(self, x, cond):
        out = self.blocks[0](x)
        # FiLM conditioning: h' = gamma * h + beta
        cond_params = self.cond_encoder(cond)
        gamma, beta = cond_params.chunk(2, dim=-1)
        gamma = gamma.unsqueeze(-1)
        beta = beta.unsqueeze(-1)
        out = gamma * out + beta
        out = self.blocks[1](out)
        return out + self.residual_conv(x)


class ConditionalUnet1D(nn.Module):
    """1D U-Net with FiLM conditioning at every residual block."""
    def __init__(self, input_dim=2, global_cond_dim=260, 
                 down_dims=(256, 512, 1024), diffusion_step_embed_dim=128):
        super().__init__()
        self.diffusion_step_encoder = nn.Sequential(
            SinusoidalPosEmb(diffusion_step_embed_dim),
            nn.Linear(diffusion_step_embed_dim, diffusion_step_embed_dim * 4),
            nn.Mish(),
            nn.Linear(diffusion_step_embed_dim * 4, diffusion_step_embed_dim),
        )
        cond_dim = diffusion_step_embed_dim + global_cond_dim
        
        all_dims = [input_dim] + list(down_dims)
        self.down_modules = nn.ModuleList()
        for i in range(len(down_dims)):
            self.down_modules.append(nn.ModuleList([
                ConditionalResidualBlock1D(all_dims[i], all_dims[i+1], cond_dim),
                ConditionalResidualBlock1D(all_dims[i+1], all_dims[i+1], cond_dim),
                nn.Conv1d(all_dims[i+1], all_dims[i+1], 3, stride=2, padding=1) if i < len(down_dims)-1 else nn.Identity(),
            ]))
        
        mid_dim = down_dims[-1]
        self.mid_modules = nn.ModuleList([
            ConditionalResidualBlock1D(mid_dim, mid_dim, cond_dim),
            ConditionalResidualBlock1D(mid_dim, mid_dim, cond_dim),
        ])
        
        self.up_modules = nn.ModuleList()
        for i in range(len(down_dims)-1, -1, -1):
            out_ch = all_dims[i] if i > 0 else down_dims[0]
            self.up_modules.append(nn.ModuleList([
                ConditionalResidualBlock1D(down_dims[i]*2, out_ch, cond_dim),
                ConditionalResidualBlock1D(out_ch, out_ch, cond_dim),
                nn.ConvTranspose1d(out_ch, out_ch, 4, stride=2, padding=1) if i > 0 else nn.Identity(),
            ]))
        
        self.final_conv = nn.Sequential(
            nn.Conv1d(down_dims[0], down_dims[0], 3, padding=1),
            nn.GroupNorm(8, down_dims[0]),
            nn.Mish(),
            nn.Conv1d(down_dims[0], input_dim, 1),
        )
    
    def forward(self, sample, timestep, local_cond=None, global_cond=None):
        timestep_emb = self.diffusion_step_encoder(timestep)
        if global_cond is not None:
            cond = torch.cat([timestep_emb, global_cond], dim=-1)
        else:
            cond = timestep_emb
        
        x = sample
        skips = []
        for res1, res2, downsample in self.down_modules:
            x = res1(x, cond)
            x = res2(x, cond)
            skips.append(x)
            x = downsample(x)
        
        for mid_block in self.mid_modules:
            x = mid_block(x, cond)
        
        for res1, res2, upsample in self.up_modules:
            skip = skips.pop()
            if x.shape[-1] != skip.shape[-1]:
                x = F.interpolate(x, size=skip.shape[-1], mode='nearest')
            x = torch.cat([x, skip], dim=1)
            x = res1(x, cond)
            x = res2(x, cond)
            x = upsample(x)
        
        return self.final_conv(x)


# Instantiate the U-Net
action_dim = 2
horizon = 16
global_cond_dim = 260

unet = ConditionalUnet1D(
    input_dim=action_dim, global_cond_dim=global_cond_dim,
    down_dims=(256, 512, 1024), diffusion_step_embed_dim=128,
).to(device)
unet.eval()

print("1D U-Net with FiLM conditioning built from scratch!")
print(f"Total parameters: {sum(p.numel() for p in unet.parameters()):,}")
print(f"\nFiLM conditioning is applied at EVERY ConditionalResidualBlock1D.")

## Building a FiLM Layer from Scratch

FiLM is surprisingly simple to implement. A single linear layer takes the conditioning vector and produces both gamma and beta.

In [ ]:
# ============================================================
# Minimal FiLM Layer Implementation
# ============================================================

class FiLMLayer(nn.Module):
    """Feature-wise Linear Modulation.
    
    Takes a conditioning vector and produces scale (gamma) and shift (beta)
    parameters that modulate the hidden features.
    """
    def __init__(self, feature_dim, cond_dim):
        super().__init__()
        # One linear layer produces both gamma and beta
        self.linear = nn.Linear(cond_dim, 2 * feature_dim)
    
    def forward(self, h, cond):
        """Apply FiLM conditioning.
        
        Args:
            h: hidden features [batch, feature_dim]
            cond: conditioning vector [batch, cond_dim]
        
        Returns:
            h': modulated features [batch, feature_dim]
        """
        params = self.linear(cond)          # [batch, 2 * feature_dim]
        gamma, beta = params.chunk(2, dim=-1)  # Each: [batch, feature_dim]
        return gamma * h + beta             # h' = gamma * h + beta


# ============================================================
# Concrete numerical example
# ============================================================
print("=" * 60)
print("FiLM: Concrete Calculation")
print("=" * 60)

h = np.array([0.5, -0.3, 0.8])
gamma = np.array([2.0, 0.1, 1.5])
beta = np.array([0.0, 0.0, 0.3])

h_prime = gamma * h + beta

print(f"\nh (hidden features):   {h}")
print(f"gamma (scale):          {gamma}")
print(f"beta (shift):           {beta}")
print(f"")
print(f"h' = gamma * h + beta:")
for i in range(3):
    print(f"  h'[{i}] = {gamma[i]:.1f} * {h[i]:.1f} + {beta[i]:.1f} = {h_prime[i]:.2f}")
print(f"\nh' (result):           {h_prime}")

print("\n" + "-" * 60)
print("Interpretation:")
print(f"  Feature 0: AMPLIFIED (gamma=2.0) -- {h[0]:.1f} -> {h_prime[0]:.2f}")
print(f"  Feature 1: SUPPRESSED (gamma=0.1) -- {h[1]:.1f} -> {h_prime[1]:.2f}")
print(f"  Feature 2: SCALED + SHIFTED -- {h[2]:.1f} -> {h_prime[2]:.2f}")

# ============================================================
# Test our FiLM layer with actual tensors
# ============================================================
print("\n" + "=" * 60)
print("FiLM Layer: Forward Pass")
print("=" * 60)

feature_dim = 64
cond_dim = 128
film = FiLMLayer(feature_dim, cond_dim)

h_tensor = torch.randn(1, feature_dim)
cond_tensor = torch.randn(1, cond_dim)

h_prime_tensor = film(h_tensor, cond_tensor)

print(f"\nh shape:       {list(h_tensor.shape)}")
print(f"cond shape:    {list(cond_tensor.shape)}")
print(f"h' shape:      {list(h_prime_tensor.shape)}")
print(f"\nh range:       [{h_tensor.min():.3f}, {h_tensor.max():.3f}]")
print(f"h' range:      [{h_prime_tensor.min():.3f}, {h_prime_tensor.max():.3f}]")
print(f"\nParameters in FiLM linear: {sum(p.numel() for p in film.parameters()):,}")
print(f"  = cond_dim * 2 * feature_dim + 2 * feature_dim")
print(f"  = {cond_dim} * {2 * feature_dim} + {2 * feature_dim} = {cond_dim * 2 * feature_dim + 2 * feature_dim}")

In [ ]:
# ============================================================
# Extract FiLM from our U-Net
# ============================================================

print("=" * 60)
print("Finding FiLM Layers in the U-Net")
print("=" * 60)

# Search for FiLM conditioning layers (cond_encoder within ConditionalResidualBlock1D)
film_layers = []
for name, module in unet.named_modules():
    if 'cond_encoder' in name and isinstance(module, nn.Sequential):
        film_layers.append((name, module))

print(f"\nFound {len(film_layers)} FiLM conditioning layers (cond_encoder)")
for name, module in film_layers:
    # Get the Linear layer inside the Sequential
    for sub_name, sub_module in module.named_modules():
        if isinstance(sub_module, nn.Linear):
            print(f"  {name}.{sub_name}: Linear({sub_module.in_features} -> {sub_module.out_features})")
            print(f"    -> produces {sub_module.out_features // 2} gamma + {sub_module.out_features // 2} beta values")

# Show the ConditionalResidualBlock1D structure
print("\n" + "=" * 60)
print("ConditionalResidualBlock1D Structure")
print("=" * 60)

for name, module in unet.named_modules():
    if isinstance(module, ConditionalResidualBlock1D):
        print(f"\n--- {name} ---")
        print(module)
        print("\nThis block contains the FiLM conditioning mechanism!")
        print("The 'cond_encoder' sub-module generates gamma and beta.")
        break

In [ ]:
# ============================================================
# Visualize: h -> h' transformation with FiLM
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.patch.set_facecolor('#1A1915')
fig.suptitle('FiLM Conditioning: h\' = gamma * h + beta', fontsize=14,
             fontweight='bold', color='white')

# Use concrete values for a small example
n_features = 8
np.random.seed(42)
h_vals = np.random.randn(n_features) * 0.5
gamma_vals = np.random.uniform(0.2, 2.5, n_features)
beta_vals = np.random.randn(n_features) * 0.3
h_prime_vals = gamma_vals * h_vals + beta_vals

x_pos = np.arange(n_features)
bar_width = 0.6

# Panel 1: Original features h
ax = axes[0]
ax.set_facecolor('#2A2A25')
bars = ax.bar(x_pos, h_vals, bar_width, color='#5B8CB8', alpha=0.85, edgecolor='white', linewidth=0.5)
ax.set_title('Original Features (h)', color='white', fontsize=12)
ax.set_xlabel('Feature Index', color='white')
ax.set_ylabel('Value', color='white')
ax.tick_params(colors='white')
ax.axhline(y=0, color='white', linestyle='-', alpha=0.3)
ax.set_ylim(-2.5, 2.5)
ax.grid(True, alpha=0.15, axis='y')
for spine in ax.spines.values():
    spine.set_color('gray')
# Annotate values
for i, v in enumerate(h_vals):
    ax.text(i, v + 0.1 * np.sign(v), f'{v:.2f}', ha='center', va='bottom' if v >= 0 else 'top',
            fontsize=7, color='white')

# Panel 2: Gamma and Beta parameters
ax = axes[1]
ax.set_facecolor('#2A2A25')
bars_g = ax.bar(x_pos - 0.15, gamma_vals, 0.3, color='#D97757', alpha=0.85,
                edgecolor='white', linewidth=0.5, label='gamma (scale)')
bars_b = ax.bar(x_pos + 0.15, beta_vals, 0.3, color='#9B7EC8', alpha=0.85,
                edgecolor='white', linewidth=0.5, label='beta (shift)')
ax.set_title('FiLM Parameters', color='white', fontsize=12)
ax.set_xlabel('Feature Index', color='white')
ax.set_ylabel('Value', color='white')
ax.tick_params(colors='white')
ax.axhline(y=0, color='white', linestyle='-', alpha=0.3)
ax.axhline(y=1, color='#D97757', linestyle='--', alpha=0.3, label='gamma=1 (no scale)')
ax.legend(fontsize=8, loc='upper right')
ax.grid(True, alpha=0.15, axis='y')
for spine in ax.spines.values():
    spine.set_color('gray')

# Panel 3: Modulated features h'
ax = axes[2]
ax.set_facecolor('#2A2A25')
# Show original as ghost bars
ax.bar(x_pos, h_vals, bar_width, color='#5B8CB8', alpha=0.2, edgecolor='#5B8CB8',
       linewidth=1, linestyle='--', label='Original h')
bars = ax.bar(x_pos, h_prime_vals, bar_width, color='#7DA488', alpha=0.85,
              edgecolor='white', linewidth=0.5, label="Modulated h'")
ax.set_title("Conditioned Features (h' = gamma*h + beta)", color='white', fontsize=12)
ax.set_xlabel('Feature Index', color='white')
ax.set_ylabel('Value', color='white')
ax.tick_params(colors='white')
ax.axhline(y=0, color='white', linestyle='-', alpha=0.3)
ax.set_ylim(-2.5, 2.5)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.15, axis='y')
for spine in ax.spines.values():
    spine.set_color('gray')
# Annotate values
for i, v in enumerate(h_prime_vals):
    ax.text(i, v + 0.1 * np.sign(v), f'{v:.2f}', ha='center', va='bottom' if v >= 0 else 'top',
            fontsize=7, color='white')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Compare: WITH conditioning vs WITHOUT conditioning (zero)
# ============================================================

with torch.no_grad():
    # Synthetic observation conditioning (simulates ResNet18 + state output)
    real_cond = torch.randn(1, global_cond_dim).to(device)
    
    # Zero conditioning (no information)
    zero_cond = torch.zeros_like(real_cond)
    
    # Same noisy input for both
    noisy_input = torch.randn(1, 2, 16).to(device)
    timestep = torch.tensor([500]).to(device)
    
    # Forward pass with real conditioning
    output_with_cond = unet(noisy_input, timestep, local_cond=None, global_cond=real_cond)
    
    # Forward pass with zero conditioning
    output_no_cond = unet(noisy_input, timestep, local_cond=None, global_cond=zero_cond)

# Convert to numpy
with_cond_np = output_with_cond.squeeze(0).cpu().numpy()   # [2, 16]
no_cond_np = output_no_cond.squeeze(0).cpu().numpy()       # [2, 16]
input_np = noisy_input.squeeze(0).cpu().numpy()            # [2, 16]

print("=" * 60)
print("Effect of Conditioning on U-Net Output")
print("=" * 60)
print(f"\nReal conditioning shape: {list(real_cond.shape)}")
print(f"Real conditioning norm:  {real_cond.norm():.3f}")
print(f"Zero conditioning norm:  {zero_cond.norm():.3f}")
print(f"\nOutput difference (L2): {(output_with_cond - output_no_cond).norm():.4f}")
print(f"Output WITH cond range:  [{with_cond_np.min():.3f}, {with_cond_np.max():.3f}]")
print(f"Output NO cond range:    [{no_cond_np.min():.3f}, {no_cond_np.max():.3f}]")

# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#1A1915')
fig.suptitle('U-Net Output: With Conditioning vs Without (Zero) Conditioning',
             fontsize=13, fontweight='bold', color='white')

t_axis = np.arange(16)
dim_names = ['X Action', 'Y Action']

for dim in range(2):
    ax = axes[dim]
    ax.set_facecolor('#2A2A25')
    
    ax.plot(t_axis, with_cond_np[dim], 'o-', color='#D97757', linewidth=2.5,
            markersize=5, label='WITH conditioning', zorder=3)
    ax.plot(t_axis, no_cond_np[dim], 's--', color='#5B8CB8', linewidth=2,
            markersize=5, label='WITHOUT conditioning (zeros)', zorder=2)
    
    # Shade the difference
    ax.fill_between(t_axis, with_cond_np[dim], no_cond_np[dim],
                    alpha=0.15, color='#C4956A', label='Difference')
    
    ax.set_title(f'{dim_names[dim]}', color='white', fontsize=12)
    ax.set_xlabel('Horizon Step', color='white')
    ax.set_ylabel('Predicted Value', color='white')
    ax.legend(fontsize=9)
    ax.tick_params(colors='white')
    ax.grid(True, alpha=0.2)
    for spine in ax.spines.values():
        spine.set_color('gray')

plt.tight_layout()
plt.show()

print("\nKey insight: The shaded region shows how much FiLM conditioning changes")
print("the U-Net output. Without conditioning, the U-Net produces a generic")
print("denoising prediction. With conditioning, it steers toward the correct")
print("action trajectory for THIS specific observation.")

In [ ]:
# ============================================================
# Extract gamma and beta from ALL FiLM layers for one input
# ============================================================

print("=" * 60)
print("FiLM Parameters Across All U-Net Layers")
print("=" * 60)

# Hook into the cond_encoder layers to capture gamma and beta
film_data = {}

def film_hook(name):
    def hook_fn(module, input, output):
        if isinstance(output, torch.Tensor):
            out_np = output.detach().cpu().numpy().flatten()
            mid = len(out_np) // 2
            film_data[name] = {
                'gamma': out_np[:mid],
                'beta': out_np[mid:],
                'full': out_np
            }
    return hook_fn

hooks = []
for name, module in unet.named_modules():
    if 'cond_encoder' in name and isinstance(module, nn.Sequential):
        hooks.append(module.register_forward_hook(film_hook(name)))

print(f"Hooked {len(hooks)} FiLM cond_encoder layers")

# Run forward pass with real conditioning
with torch.no_grad():
    real_cond = torch.randn(1, global_cond_dim).to(device)
    noisy_input = torch.randn(1, 2, 16).to(device)
    timestep = torch.tensor([500]).to(device)
    output = unet(noisy_input, timestep, local_cond=None, global_cond=real_cond)

# Clean up hooks
for h in hooks:
    h.remove()

print(f"Captured FiLM parameters from {len(film_data)} layers")

if film_data:
    # Visualize gamma and beta distributions across layers
    n_layers = len(film_data)
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    fig.patch.set_facecolor('#1A1915')
    fig.suptitle('FiLM Gamma and Beta Values Across U-Net Layers',
                 fontsize=14, fontweight='bold', color='white')

    layer_names = list(film_data.keys())
    # Create short names for readability
    short_names = []
    for n in layer_names:
        parts = n.split('.')
        # e.g., "down_modules.0.0.cond_encoder" -> "down_0.res0"
        if 'down' in n:
            short_names.append(f"down.{parts[1]}.res{parts[2]}")
        elif 'mid' in n:
            short_names.append(f"mid.{parts[1]}")
        elif 'up' in n:
            short_names.append(f"up.{parts[1]}.res{parts[2]}")
        else:
            short_names.append('.'.join(parts[-3:]))

    # Gamma statistics
    ax = axes[0]
    ax.set_facecolor('#2A2A25')
    gamma_means = [film_data[n]['gamma'].mean() for n in layer_names]
    gamma_stds = [film_data[n]['gamma'].std() for n in layer_names]
    x = np.arange(n_layers)
    ax.bar(x, gamma_means, yerr=gamma_stds, color='#D97757', alpha=0.8,
           edgecolor='white', linewidth=0.5, capsize=3)
    ax.axhline(y=1.0, color='white', linestyle='--', alpha=0.4, label='gamma=1 (no modulation)')
    ax.set_ylabel('Gamma (scale)', color='white')
    ax.set_title('Scale Parameters (gamma) per Layer', color='white')
    ax.set_xticks(x)
    ax.set_xticklabels(short_names, rotation=45, ha='right', fontsize=7, color='white')
    ax.tick_params(colors='white')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.15, axis='y')
    for spine in ax.spines.values():
        spine.set_color('gray')

    # Beta statistics
    ax = axes[1]
    ax.set_facecolor('#2A2A25')
    beta_means = [film_data[n]['beta'].mean() for n in layer_names]
    beta_stds = [film_data[n]['beta'].std() for n in layer_names]
    ax.bar(x, beta_means, yerr=beta_stds, color='#9B7EC8', alpha=0.8,
           edgecolor='white', linewidth=0.5, capsize=3)
    ax.axhline(y=0.0, color='white', linestyle='--', alpha=0.4, label='beta=0 (no shift)')
    ax.set_ylabel('Beta (shift)', color='white')
    ax.set_title('Shift Parameters (beta) per Layer', color='white')
    ax.set_xticks(x)
    ax.set_xticklabels(short_names, rotation=45, ha='right', fontsize=7, color='white')
    ax.tick_params(colors='white')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.15, axis='y')
    for spine in ax.spines.values():
        spine.set_color('gray')

    plt.tight_layout()
    plt.show()

    print("\nGamma near 1.0 = minimal scaling. Beta near 0.0 = minimal shift.")
    print("Layers with large gamma/beta deviations are where conditioning has the most impact.")
    print("\n(Note: With random weights, the FiLM parameters are not yet meaningful.")
    print("After training, specific layers would learn to apply stronger modulation.)")

## Key Takeaways

1. **FiLM is simple but powerful**: Just `h' = gamma * h + beta`, where gamma and beta come from the conditioning signal
2. **It appears at every ResBlock**: The U-Net applies FiLM conditioning at every residual block, meaning the observation influences the denoising process at every scale
3. **Gamma = attention, Beta = bias**: Gamma controls which features to amplify/suppress; Beta shifts the baseline activation
4. **Without conditioning, the U-Net is blind**: Zero conditioning produces a generic output; real conditioning steers toward the correct action trajectory

## TODO Exercise

What happens if you set all gamma=1 and beta=0 (identity FiLM)? 

- Hook into the FiLM layers and override their outputs to always be `gamma=1, beta=0`
- Run the full denoising loop with this "disabled" FiLM
- Compare the resulting trajectory to one with normal FiLM conditioning
- How does this affect denoising quality? Does the model still produce coherent actions?